In [ ]:
%pip install --upgrade --quiet docx2txt langchain-community langchain-classic
%pip install -qU langchain-text-splitters
%pip install -qU "langchain-chroma>=0.1.2"

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500, # 하나의 청크가 가지는 토큰 수
    chunk_overlap=200, # 청크 간의 겹치는 토큰 수
)

loader = Docx2txtLoader('./tax_with_markdown.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

In [187]:
# chunk_size=1500, chunk_overlap=200 일때 제55조(세율) 문서 - document_list[52]

# 벡터 디비에서 문서 검색이 안되어 chunk_size, chunk_overlap 조정
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
document_list = loader.load_and_split(text_splitter=text_splitter)

In [ ]:
question = "연봉 5천만원 직장인의 소득세는 얼마인가요?"

In [ ]:
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv

load_dotenv()

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [188]:
from langchain_chroma import Chroma

database = Chroma.from_documents(
    documents=document_list,
    embedding=embeddings,
    collection_name='chroma_tax_markdown12',
    persist_directory='./db/chroma'
)

In [189]:
retriever = database.as_retriever(search_kwargs={"k": 3})
retrieved_docs = retriever.invoke(question)

In [190]:
retrieved_docs

[Document(id='141fbfdb-20ca-40c7-bcf4-072dc64202f0', metadata={'source': './tax_with_markdown.docx'}, page_content='[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n| 종합소득 과세표준          | 세율                                         |\n\n|-------------------|--------------------------------------------|\n\n| 1,400만원 이하     | 과세표준의 6퍼센트                             |\n\n| 1,400만원 초과     5,000만원 이하     | 84만원 + (1,400만원을 초과하는 금액의 15퍼센트)  |\n\n| 5,000만원 초과   8,800만원 이하     | 624만원 + (5,000만원을 초과하는 금액의 24퍼센트) |\n\n| 8,800만원 초과 1억5천만원 이하    | 3,706만원 + (8,800만원을 초과하는 금액의 35퍼센트)|\n\n| 1억5천만원 초과 3억원 이하         | 3,706만원 + (1억5천만원을 초과하는 금액의 38퍼센트)|\n\n| 3억원 초과    5억원 이하         | 9,406만원 + (3억원을 초과하는 금액의 38퍼센트)   |\n\n| 5억원 초과      10억원 이하        | 1억 

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """[Identity]
        당신은 소득세법 전문가입니다.
        - [Context]을 참고해서 사용자의 질문에 답변해주세요.

        [Context]
        {context}
        """
    ),
    (
        "human",
        "[Question]: {question}"
    )
])

In [126]:
prompt.input_variables # ['context', 'question']

['context', 'question']

In [181]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o')

In [191]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt},
)

In [192]:
# LangChain 내부 매핑 로직
# prompt.input_variables: question 가 포함되어야 함
# qa_chain.input_keys: query 가 포함되어야 함

ai_message = qa_chain.invoke({ "query": query })

In [ ]:
ai_message["result"] # 따라서, 연봉 5천만원인 직장인의 소득세는 624만원입니다.

'연봉 5천만원 직장인의 소득세를 계산하기 위해서는 종합소득 과세표준에 따른 세율을 적용해야 합니다. 여기서는 소득세 계산 시 기본공제 등을 고려하지 않고 단순히 소득세율만 적용하여 계산하겠습니다.\n\n1. 과세표준이 5천만원인 경우:\n   - 1,400만원까지는 6% 세율 적용: 1,400만원 × 6% = 84만원\n   - 1,400만원을 초과하여 5,000만원까지는 15% 세율 적용: (5,000만원 - 1,400만원) × 15% = 3,600만원 × 15% = 540만원\n\n2. 따라서, 총 소득세는:\n   - 84만원 + 540만원 = 624만원\n\n따라서, 연봉 5천만원인 직장인의 소득세는 624만원입니다. 하지만 실제 소득세 계산 시에는 기본 인적 공제, 연금보험료 공제 등 각종 소득공제를 반영하여 과세표준을 산출한 후 세금을 계산해야 하므로, 이는 근로자의 실제 세금과 다를 수 있습니다. 이를 감안하여 실제 세금은 더 적을 수 있습니다.'